In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk
df = pd.read_csv('/Users/yume/PycharmProjects/PBL2/data/reddit_dr.csv')
df["Text"] = (
    df["Title"].fillna("").astype(str)
    + " "
    + df["Text"].fillna("").astype(str)
).str.strip()
df = df.drop(columns=["Title"])

In [2]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = df["Text"].fillna("").astype(str).tolist()

X = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

y = df["Political Lean"]

# Remove missing labels
label_mask = y.notna()
X = X[label_mask.to_numpy()]
y = y[label_mask].reset_index(drop=True)

# Remove invalid embedding rows
valid_rows = np.isfinite(X).all(axis=1)
X = X[valid_rows]
y = y.iloc[valid_rows].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

classifier = LogisticRegression(
    solver="liblinear",
    C=0.1,
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 402/402 [00:14<00:00, 27.39it/s]


              precision    recall  f1-score   support

Conservative       0.56      0.70      0.62       907
     Liberal       0.81      0.70      0.75      1664

    accuracy                           0.70      2571
   macro avg       0.68      0.70      0.68      2571
weighted avg       0.72      0.70      0.70      2571

[[ 631  276]
 [ 502 1162]]


/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [4]:
import numpy as np
import sklearn
import scipy

print("X shape:", X.shape)
print("X dtype:", X.dtype)
print("X finite:", np.isfinite(X).all())
print("X min:", X.min())
print("X max:", X.max())
print("Largest row norm:", np.linalg.norm(X, axis=1).max())

print("\nLabels:")
print(y.value_counts(dropna=False))

print("\nVersions:")
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)

X shape: (12854, 384)
X dtype: float32
X finite: True
X min: -0.28252834
X max: 0.32259098
Largest row norm: 1.0000001

Labels:
Political Lean
Liberal         8319
Conservative    4535
Name: count, dtype: int64

Versions:
NumPy: 2.2.6
SciPy: 1.15.3
scikit-learn: 1.7.2


In [8]:
%pip install --upgrade numpy scipy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [5]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

classifier = LinearSVC(
    C=0.1,
    class_weight="balanced",
    random_state=42
)

classifier.fit(X_train.astype(np.float64), y_train)

y_pred = classifier.predict(X_test.astype(np.float64))

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

Conservative       0.57      0.70      0.63       907
     Liberal       0.81      0.71      0.76      1664

    accuracy                           0.71      2571
   macro avg       0.69      0.70      0.69      2571
weighted avg       0.73      0.71      0.71      2571

[[ 634  273]
 [ 482 1182]]


/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [6]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    C=0.1,
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

classifier.fit(
    X_train.astype(np.float64),
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.1
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'liblinear'
,max_iter,3000
,multi_class,'deprecated'


In [7]:
print(classifier.n_iter_)
print(np.isfinite(classifier.coef_).all())

[4]
True


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

logistic = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    class_weight="balanced",
    max_iter=3000,
    random_state=42
)

param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    estimator=logistic,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train.astype("float64"), y_train)

print("Best C:", grid.best_params_)
print("Best CV macro F1:", grid.best_score_)

best_logistic = grid.best_estimator_

Fitting 5 folds for each of 6 candidates, totalling 30 fits


/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/yume/Pychar

Best C: {'C': 10}
Best CV macro F1: 0.6910708148207516


In [9]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = best_logistic.predict(X_test.astype("float64"))

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

Conservative       0.57      0.69      0.62       907
     Liberal       0.81      0.71      0.76      1664

    accuracy                           0.71      2571
   macro avg       0.69      0.70      0.69      2571
weighted avg       0.72      0.71      0.71      2571

[[ 630  277]
 [ 480 1184]]


/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

text_train, text_test, y_train_text, y_test_text = train_test_split(
    df["Text"].fillna(""),
    df["Political Lean"],
    test_size=0.2,
    random_state=42,
    stratify=df["Political Lean"]
)

tfidf_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            sublinear_tf=True,
            max_features=50000
        )
    ),
    (
        "classifier",
        LogisticRegression(
            solver="liblinear",
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        )
    )
])

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10]
}

tfidf_search = GridSearchCV(
    tfidf_pipeline,
    param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

tfidf_search.fit(text_train, y_train_text)

print("Best parameters:", tfidf_search.best_params_)

tfidf_pred = tfidf_search.predict(text_test)

print(classification_report(y_test_text, tfidf_pred))

Best parameters: {'classifier__C': 1}
              precision    recall  f1-score   support

Conservative       0.68      0.69      0.69       907
     Liberal       0.83      0.83      0.83      1664

    accuracy                           0.78      2571
   macro avg       0.76      0.76      0.76      2571
weighted avg       0.78      0.78      0.78      2571



In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack, csr_matrix
embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    max_features=50000,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(df["Text"].fillna(""))

svd = TruncatedSVD(
    n_components=300,
    random_state=42
)

X_tfidf_reduced = svd.fit_transform(X_tfidf)

X_combined = np.hstack([
    embeddings.astype(np.float64),
    X_tfidf_reduced.astype(np.float64)
])

print(X_combined.shape)

Batches: 100%|██████████| 402/402 [00:12<00:00, 31.53it/s] 


(12854, 684)


/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined,
    df["Political Lean"],
    test_size=0.2,
    random_state=42,
    stratify=df["Political Lean"]
)

In [13]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

svm = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm_grid = GridSearchCV(
    svm,
    param_grid={"C": [0.001, 0.01, 0.1, 1, 10]},
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

svm_grid.fit(X_train, y_train)

print("Best C:", svm_grid.best_params_)
print("Best CV macro F1:", svm_grid.best_score_)

svm_pred = svm_grid.predict(X_test)

print(classification_report(y_test, svm_pred))

/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/yume/Pychar

Best C: {'C': 1}
Best CV macro F1: 0.7156298971340026
              precision    recall  f1-score   support

Conservative       0.61      0.72      0.66       907
     Liberal       0.83      0.75      0.79      1664

    accuracy                           0.74      2571
   macro avg       0.72      0.73      0.72      2571
weighted avg       0.75      0.74      0.74      2571



/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [14]:
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

combined_features = FeatureUnion([
    (
        "word_tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            sublinear_tf=True,
            max_features=100000,
            strip_accents="unicode"
        )
    ),
    (
        "char_tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=100000,
            sublinear_tf=True
        )
    )
])

svm_pipeline = Pipeline([
    ("features", combined_features),
    (
        "classifier",
        LinearSVC(
            class_weight="balanced",
            random_state=42,
            max_iter=5000
        )
    )
])

param_grid = {
    "classifier__C": [0.01, 0.05, 0.1, 0.5, 1, 2, 5]
}

svm_search = GridSearchCV(
    svm_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=2
)

svm_search.fit(text_train, y_train_text)

print("Best parameters:", svm_search.best_params_)
print("Best CV macro F1:", svm_search.best_score_)

svm_pred = svm_search.predict(text_test)

print(classification_report(y_test_text, svm_pred))
print(confusion_matrix(y_test_text, svm_pred))

Fitting 5 folds for each of 7 candidates, totalling 35 fits
Best parameters: {'classifier__C': 0.1}
Best CV macro F1: 0.7495042069280922
              precision    recall  f1-score   support

Conservative       0.68      0.67      0.68       907
     Liberal       0.82      0.83      0.83      1664

    accuracy                           0.77      2571
   macro avg       0.75      0.75      0.75      2571
weighted avg       0.77      0.77      0.77      2571

[[ 610  297]
 [ 283 1381]]


In [15]:
from sklearn.model_selection import RandomizedSearchCV

tfidf_search = RandomizedSearchCV(
    tfidf_pipeline,
    param_distributions=param_grid,
    n_iter=30,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=2
)

tfidf_search.fit(text_train, y_train_text)

/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 7 is smaller than n_iter=30. Running 7 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 7 candidates, totalling 35 fits


,estimator,Pipeline(step...liblinear'))])
,param_distributions,"{'classifier__C': [0.01, 0.05, ...]}"
,n_iter,30
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [16]:
print("Best parameters:")
print(tfidf_search.best_params_)

print("\nBest cross-validation macro F1:")
print(tfidf_search.best_score_)

Best parameters:
{'classifier__C': 1}

Best cross-validation macro F1:
0.7439804204449487


In [17]:
best_model = tfidf_search.best_estimator_

vectorizer = best_model.named_steps["tfidf"]
classifier = best_model.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()
coefficients = classifier.coef_[0]

top_conservative = coefficients.argsort()[-25:][::-1]
top_liberal = coefficients.argsort()[:25]

print("Top Conservative-associated features:")
for i in top_conservative:
    print(feature_names[i], coefficients[i])

print("\nTop Liberal-associated features:")
for i in top_liberal:
    print(feature_names[i], coefficients[i])

Top Conservative-associated features:
women 3.7351794901093247
workers 3.556684171083164
feminist 2.378346090599837
dsa 2.2792660655195167
democracy 2.2738663989771633
democratic 2.219827887300969
cuba 2.1748675779987705
opinion 2.173058715216151
solidarity 2.1595410623691187
plan 2.0977658458751383
imperialism 1.9827793606448805
feminism 1.976171627879896
election 1.9289563075882612
jan 1.8849488115135715
socialist 1.761536051786338
progressive 1.7475091188556022
coup 1.7196503532576497
strike 1.708087213863119
party 1.7027983438811796
social 1.7026722564532184
sexual 1.590169577246793
analysis 1.5365078116817983
struggle 1.4753662341391287
far right 1.4294127982749556
wing 1.4258557375892358

Top Liberal-associated features:
libertarian -4.7754701648706135
desantis -3.65086920123022
trudeau -3.3819974283275274
putin -3.3091851352125397
libertarians -3.271427564690417
ancap -3.20337465527055
ukraine -3.202615099676658
government -3.1228464979259343
capitalism -3.049986354829893
russia

In [18]:
import nltk

nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/yume/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [23]:
# %% imports
import numpy as np
import pandas as pd
import nltk

from nltk.sentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
sia = SentimentIntensityAnalyzer()

sentiment_features = (
    df["Text"]
    .fillna("")
    .astype(str)
    .apply(sia.polarity_scores)
    .apply(pd.Series)
)

sentiment_features.head()

,neg,neu,pos,compound
0,0.058,0.847,0.095,0.1027
1,0.000,1.000,0.000,0.0000
2,0.000,1.000,0.000,0.0000
3,0.341,0.659,0.000,-0.4767
4,0.000,1.000,0.000,0.0000


In [24]:
print(sentiment_features.shape)
print(embeddings.shape)

(12854, 4)
(12854, 384)


In [38]:
#combine embeddings and sentiment features
# Ensure embeddings are regular numeric values
embeddings_clean = np.asarray(embeddings, dtype=np.float64)

# Convert sentiment dataframe into a NumPy matrix
sentiment_array = sentiment_features[
    ["neg", "neu", "pos", "compound"]
].to_numpy(dtype=np.float64)

# Combine them column-wise
X = np.hstack([
    embeddings_clean,
    sentiment_array
])

y = df["Political Lean"].copy()

print("Combined feature shape:", X.shape)
print("Label shape:", y.shape)

ModuleNotFoundError: No module named 'vaderSentiment'

In [26]:
#clean
label_mask = y.notna()

X = X[label_mask.to_numpy()]
y = y[label_mask].reset_index(drop=True)

valid_rows = np.isfinite(X).all(axis=1)

X = X[valid_rows]
y = y.iloc[valid_rows].reset_index(drop=True)

print("Final X shape:", X.shape)
print(y.value_counts())

Final X shape: (12854, 388)
Political Lean
Liberal         8319
Conservative    4535
Name: count, dtype: int64


In [27]:
#split train test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
classifier_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            solver="liblinear",
            penalty="l2",
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        )
    )
])

In [29]:
param_grid = {
    "classifier__C": [
        0.001,
        0.01,
        0.05,
        0.1,
        0.5,
        1,
        5,
        10
    ]
}

grid_search = GridSearchCV(
    estimator=classifier_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/yume/Pychar

,estimator,Pipeline(step...liblinear'))])
,param_grid,"{'classifier__C': [0.001, 0.01, ...]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [30]:
print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation macro F1:")
print(grid_search.best_score_)

Best parameters:
{'classifier__C': 0.01}

Best cross-validation macro F1:
0.6915034102431152


In [31]:
best_classifier = grid_search.best_estimator_

In [32]:
y_pred = best_classifier.predict(X_test)

/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [33]:
print("Test accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Test accuracy:
0.7067288992609879

Classification report:
              precision    recall  f1-score   support

Conservative       0.57      0.70      0.63       907
     Liberal       0.81      0.71      0.76      1664

    accuracy                           0.71      2571
   macro avg       0.69      0.71      0.69      2571
weighted avg       0.73      0.71      0.71      2571


Confusion matrix:
[[ 638  269]
 [ 485 1179]]


In [34]:
X_embeddings_only = embeddings_clean[label_mask.to_numpy()]
X_embeddings_only = X_embeddings_only[valid_rows]

X_train_emb, X_test_emb, y_train_emb, y_test_emb = train_test_split(
    X_embeddings_only,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

embedding_only_search = GridSearchCV(
    estimator=Pipeline([
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                solver="liblinear",
                class_weight="balanced",
                max_iter=3000,
                random_state=42
            )
        )
    ]),
    param_grid={
        "classifier__C": [
            0.001,
            0.01,
            0.05,
            0.1,
            0.5,
            1,
            5,
            10
        ]
    },
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

embedding_only_search.fit(X_train_emb, y_train_emb)

embedding_only_pred = embedding_only_search.predict(X_test_emb)

print("Embeddings-only accuracy:")
print(accuracy_score(y_test_emb, embedding_only_pred))

print("\nEmbeddings-only report:")
print(classification_report(y_test_emb, embedding_only_pred))

/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/yume/Pychar

Embeddings-only accuracy:
0.7040062232594321

Embeddings-only report:
              precision    recall  f1-score   support

Conservative       0.57      0.69      0.62       907
     Liberal       0.81      0.71      0.76      1664

    accuracy                           0.70      2571
   macro avg       0.69      0.70      0.69      2571
weighted avg       0.72      0.70      0.71      2571



/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [35]:
comparison = pd.DataFrame({
    "Model": [
        "Embeddings only",
        "Embeddings + sentiment"
    ],
    "Accuracy": [
        accuracy_score(y_test_emb, embedding_only_pred),
        accuracy_score(y_test, y_pred)
    ],
    "CV Macro F1": [
        embedding_only_search.best_score_,
        grid_search.best_score_
    ]
})

comparison

,Model,Accuracy,CV Macro F1
0,Embeddings only,0.704006,0.689506
1,Embeddings + sentiment,0.706729,0.691503


In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

sia = SentimentIntensityAnalyzer()

sentiment_features = (
    df["Text"]
    .fillna("")
    .astype(str)
    .apply(sia.polarity_scores)
    .apply(pd.Series)
)

df[["neg", "neu", "pos", "compound"]] = sentiment_features[
    ["neg", "neu", "pos", "compound"]
]
print(df.columns)
print(df[["neg", "neu", "pos", "compound"]].head())
feature_columns = ["Text", "neg", "neu", "pos", "compound"]

X_train, X_test, y_train, y_test = train_test_split(
    df[feature_columns],
    df["Political Lean"],
    test_size=0.2,
    random_state=42,
    stratify=df["Political Lean"]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.95,
                sublinear_tf=True,
                max_features=50000
            ),
            "Text"
        ),
        (
            "sentiment",
            StandardScaler(),
            ["neg", "neu", "pos", "compound"]
        )
    ]
)

tfidf_sentiment_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            solver="liblinear",
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        )
    )
])

search = GridSearchCV(
    tfidf_sentiment_pipeline,
    param_grid={
        "classifier__C": [0.01, 0.1, 1, 10]
    },
    scoring="f1_macro",
    cv=5,
    n_jobs=-1
)

search.fit(X_train, y_train)

pred = search.predict(X_test)

print("Best parameters:", search.best_params_)
print("Best CV macro F1:", search.best_score_)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

Index(['Political Lean', 'Score', 'Id', 'Subreddit', 'URL', 'Num of Comments',
       'Text', 'Date Created', 'neg', 'neu', 'pos', 'compound'],
      dtype='object')
     neg    neu    pos  compound
0  0.029  0.902  0.068    0.3455
1  0.000  1.000  0.000    0.0000
2  0.000  1.000  0.000    0.0000
3  0.341  0.659  0.000   -0.4767
4  0.000  1.000  0.000    0.0000
Best parameters: {'classifier__C': 1}
Best CV macro F1: 0.7442705575390407
Accuracy: 0.7736289381563594
              precision    recall  f1-score   support

Conservative       0.68      0.68      0.68       907
     Liberal       0.83      0.82      0.82      1664

    accuracy                           0.77      2571
   macro avg       0.75      0.75      0.75      2571
weighted avg       0.77      0.77      0.77      2571

[[ 618  289]
 [ 293 1371]]


## Transformer-based sentiment analysis

In [40]:
%pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [19]:
from transformers import pipeline
import pandas as pd

sentiment_model = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    top_k=None,
    truncation=True,
    max_length=512
)

def get_transformer_sentiment(text):
    results = sentiment_model(str(text))[0]

    scores = {
        item["label"].lower(): item["score"]
        for item in results
    }

    return pd.Series({
        "roberta_negative": scores.get("negative", 0),
        "roberta_neutral": scores.get("neutral", 0),
        "roberta_positive": scores.get("positive", 0)
    })

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 60226.83it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
test_result = sentiment_model(
    "I am extremely happy that this finally worked."
)

print(test_result)

[[{'label': 'positive', 'score': 0.9839354753494263}, {'label': 'neutral', 'score': 0.011693735606968403}, {'label': 'negative', 'score': 0.004370782524347305}]]


In [22]:
#batched processing
texts = df["Text"].fillna("").astype(str).tolist()

results = sentiment_model(
    texts,
    batch_size=16,
    truncation=True,
    max_length=512
)

rows = []

for result in results:
    scores = {
        item["label"].lower(): item["score"]
        for item in result
    }

    rows.append({
        "roberta_negative": scores.get("negative", 0),
        "roberta_neutral": scores.get("neutral", 0),
        "roberta_positive": scores.get("positive", 0)
    })

roberta_sentiment = pd.DataFrame(rows)

df = pd.concat(
    [df.reset_index(drop=True),
     roberta_sentiment],
    axis=1
)

In [23]:
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    truncation=True,
    max_length=512
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 46091.25it/s]


In [24]:
emotion_results = emotion_model(
    texts,
    batch_size=16,
    truncation=True,
    max_length=512
)

emotion_rows = []

for result in emotion_results:
    emotion_rows.append({
        f"emotion_{item['label'].lower()}": item["score"]
        for item in result
    })

emotion_df = pd.DataFrame(emotion_rows)

df = pd.concat(
    [df.reset_index(drop=True),
     emotion_df.reset_index(drop=True)],
    axis=1
)

In [25]:
import numpy as np

text_series = df["Text"].fillna("").astype(str)

df["word_count"] = text_series.str.split().str.len()
df["char_count"] = text_series.str.len()

df["avg_word_length"] = (
    text_series.apply(
        lambda x: np.mean([len(w) for w in x.split()])
        if x.split() else 0
    )
)

df["exclamation_count"] = text_series.str.count("!")
df["question_count"] = text_series.str.count(r"\?")
df["uppercase_count"] = text_series.apply(
    lambda x: sum(1 for c in x if c.isupper())
)

df["uppercase_ratio"] = (
    df["uppercase_count"] /
    df["char_count"].replace(0, 1)
)

df["url_count"] = text_series.str.count(r"http\S+")
df["mention_count"] = text_series.str.count(r"@\w+")
df["hashtag_count"] = text_series.str.count(r"#\w+")

In [26]:
df["sentence_count"] = text_series.str.count(r"[.!?]+").clip(lower=1)

df["avg_sentence_length"] = (
    df["word_count"] /
    df["sentence_count"]
)

In [27]:
def type_token_ratio(text):
    tokens = str(text).lower().split()

    if not tokens:
        return 0

    return len(set(tokens)) / len(tokens)

df["type_token_ratio"] = text_series.apply(type_token_ratio)

In [28]:
%pip install textblob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.0/625.0 kB 3.3 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [29]:
from textblob import TextBlob

df["textblob_polarity"] = text_series.apply(
    lambda x: TextBlob(x).sentiment.polarity
)

df["textblob_subjectivity"] = text_series.apply(
    lambda x: TextBlob(x).sentiment.subjectivity
)

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

topic_vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=3,
    max_df=0.95,
    ngram_range=(1, 2),
    max_features=50000
)

X_topic_tfidf = topic_vectorizer.fit_transform(
    df["Text"].fillna("")
)

svd = TruncatedSVD(
    n_components=50,
    random_state=42
)

topic_features = svd.fit_transform(X_topic_tfidf)

topic_df = pd.DataFrame(
    topic_features,
    columns=[f"topic_{i}" for i in range(topic_features.shape[1])]
)

df = pd.concat(
    [df.reset_index(drop=True),
     topic_df.reset_index(drop=True)],
    axis=1
)

/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/yume/PycharmProjects/PBL2/.venv1/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat


In [31]:
numeric_features = [
    "roberta_negative",
    "roberta_neutral",
    "roberta_positive",

    "emotion_anger",
    "emotion_disgust",
    "emotion_fear",
    "emotion_joy",
    "emotion_neutral",
    "emotion_sadness",
    "emotion_surprise",

    "word_count",
    "char_count",
    "avg_word_length",
    "avg_sentence_length",
    "exclamation_count",
    "question_count",
    "uppercase_ratio",
    "type_token_ratio",
    "root_ttr",
    "textblob_subjectivity"
]

KeyboardInterrupt: 

In [33]:
sentiment_rows = []

for result in results:
    scores = {
        item["label"].lower(): item["score"]
        for item in result
    }

    sentiment_rows.append({
        "roberta_negative": scores.get("negative", 0.0),
        "roberta_neutral": scores.get("neutral", 0.0),
        "roberta_positive": scores.get("positive", 0.0)
    })

roberta_sentiment_df = pd.DataFrame(sentiment_rows)

In [34]:
df = df.reset_index(drop=True)
roberta_sentiment_df = roberta_sentiment_df.reset_index(drop=True)

df[
    [
        "roberta_negative",
        "roberta_neutral",
        "roberta_positive"
    ]
] = roberta_sentiment_df[
    [
        "roberta_negative",
        "roberta_neutral",
        "roberta_positive"
    ]
]

In [35]:
model_df = df.dropna(
    subset=[
        "Text",
        "Political Lean",
        "roberta_negative",
        "roberta_neutral",
        "roberta_positive"
    ]
).copy()

In [36]:
feature_columns = [
    "Text",
    "roberta_negative",
    "roberta_neutral",
    "roberta_positive"
]

X = model_df[feature_columns]
y = model_df["Political Lean"]

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
sentiment_features = [
    "roberta_negative",
    "roberta_neutral",
    "roberta_positive"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=3,
                max_df=0.95,
                sublinear_tf=True,
                max_features=50000
            ),
            "Text"
        ),
        (
            "sentiment",
            StandardScaler(),
            sentiment_features
        )
    ]
)

In [41]:
roberta_tfidf_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            solver="liblinear",
            penalty="l2",
            class_weight="balanced",
            max_iter=3000,
            random_state=42
        )
    )
])

In [42]:
param_grid = {
    "preprocessor__tfidf__ngram_range": [
        (1, 2),
        (1, 3)
    ],
    "preprocessor__tfidf__min_df": [
        2,
        3,
        5
    ],
    "preprocessor__tfidf__stop_words": [
        None,
        "english"
    ],
    "classifier__C": [
        0.1,
        0.5,
        1,
        2,
        5
    ]
}

In [43]:
roberta_tfidf_search = GridSearchCV(
    estimator=roberta_tfidf_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

roberta_tfidf_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 60 candidates, totalling 300 fits


,estimator,Pipeline(step...liblinear'))])
,param_grid,"{'classifier__C': [0.1, 0.5, ...], 'preprocessor__tfidf__min_df': [2, 3, ...], 'preprocessor__tfidf__ngram_range': [(1, ...), (1, ...)], 'preprocessor__tfidf__stop_words': [None, 'english']}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('tfidf', ...), ('sentiment', ...)]"


In [44]:
print("Best parameters:")
print(roberta_tfidf_search.best_params_)

print("\nBest CV macro F1:")
print(roberta_tfidf_search.best_score_)

Best parameters:
{'classifier__C': 1, 'preprocessor__tfidf__min_df': 2, 'preprocessor__tfidf__ngram_range': (1, 2), 'preprocessor__tfidf__stop_words': 'english'}

Best CV macro F1:
0.7494377763508607


In [45]:
best_roberta_tfidf = roberta_tfidf_search.best_estimator_

y_pred = best_roberta_tfidf.predict(X_test)

In [48]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.metrics import f1_score
test_accuracy = accuracy_score(
    y_test,
    y_pred
)

test_macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print("Test accuracy:", test_accuracy)
print("Test macro F1:", test_macro_f1)

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Test accuracy: 0.7817969661610268
Test macro F1: 0.7606623915491064

Classification report:
              precision    recall  f1-score   support

Conservative       0.69      0.69      0.69       907
     Liberal       0.83      0.83      0.83      1664

    accuracy                           0.78      2571
   macro avg       0.76      0.76      0.76      2571
weighted avg       0.78      0.78      0.78      2571


Confusion matrix:
[[ 623  284]
 [ 277 1387]]


In [49]:
search_results = pd.DataFrame(
    roberta_tfidf_search.cv_results_
)

best_row = search_results.loc[
    search_results["rank_test_score"] == 1,
    [
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "params"
    ]
]

best_row

,mean_train_score,mean_test_score,std_test_score,params
25,0.90523,0.749438,0.006279,"{'classifier__C': 1, 'preprocessor__tfidf__min..."


In [50]:
import numpy as np
import pandas as pd

best_model = roberta_tfidf_search.best_estimator_

preprocessor = best_model.named_steps["preprocessor"]
classifier = best_model.named_steps["classifier"]

feature_names = preprocessor.get_feature_names_out()
coefficients = classifier.coef_[0]

feature_coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
})

feature_coef_df.head()

,feature,coefficient,abs_coefficient
0,tfidf__00,0.274798,0.274798
1,tfidf__00 pm,0.096393,0.096393
2,tfidf__000,-0.093936,0.093936
3,tfidf__000 000,-0.199277,0.199277
4,tfidf__000 american,0.069649,0.069649


In [51]:
print(classifier.classes_)

['Conservative' 'Liberal']


In [52]:
feature_coef_df["feature_clean"] = (
    feature_coef_df["feature"]
    .str.replace("tfidf__", "", regex=False)
    .str.replace("sentiment__", "", regex=False)
)

In [53]:
top_liberal = (
    feature_coef_df
    .sort_values("coefficient", ascending=False)
    .head(30)
)

top_conservative = (
    feature_coef_df
    .sort_values("coefficient", ascending=True)
    .head(30)
)

print("Top Liberal-associated features:")
display(top_liberal[["feature_clean", "coefficient"]])

print("Top Conservative-associated features:")
display(top_conservative[["feature_clean", "coefficient"]])

Top Liberal-associated features:


,feature_clean,coefficient
31886,women,3.890330
32115,workers,3.636827
10731,feminist,2.367332
7703,democratic,2.311278
19865,opinion,2.264668
7635,democracy,2.194177
8978,dsa,2.183785
7047,cuba,2.118939
26767,solidarity,2.114831
21377,plan,2.043522


Top Conservative-associated features:


,feature_clean,coefficient
16414,libertarian,-4.774152
7925,desantis,-3.701013
22986,putin,-3.384855
29520,trudeau,-3.321924
29881,ukraine,-3.318491
12217,government,-3.243465
16452,libertarians,-3.208107
4445,capitalism,-3.198888
24948,russia,-3.167766
1948,ancap,-3.114607


In [54]:
sentiment_coef_df = feature_coef_df[
    feature_coef_df["feature"].str.startswith("sentiment__")
].sort_values("coefficient", ascending=False)

sentiment_coef_df[["feature_clean", "coefficient"]]

,feature_clean,coefficient
32736,roberta_positive,0.011995
32734,roberta_negative,0.005049
32735,roberta_neutral,-0.014676


In [56]:
sentiment_summary = (
    df.groupby("Political Lean")[
        [
            "roberta_negative",
            "roberta_neutral",
            "roberta_positive"
        ]
    ]
    .agg(["mean", "std"])
)

sentiment_summary

roberta_negative           roberta_neutral            \
                           mean       std            mean       std   
Political Lean                                                        
Conservative           0.357274  0.298514        0.531486  0.263342   
Liberal                0.354693  0.308754        0.529800  0.268944   

               roberta_positive            
                           mean       std  
Political Lean                             
Conservative           0.111240  0.197684  
Liberal                0.115507  0.196247

## Moral foundation features

In [57]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

semantic_model = SentenceTransformer("all-mpnet-base-v2")

moral_prototypes = {
    "care": "This text emphasizes compassion, care, protection, and preventing harm.",
    "fairness": "This text emphasizes justice, equality, fairness, and equal treatment.",
    "loyalty": "This text emphasizes loyalty, patriotism, group solidarity, and betrayal.",
    "authority": "This text emphasizes authority, order, tradition, and respect for hierarchy.",
    "purity": "This text emphasizes purity, sanctity, morality, contamination, or degradation.",
    "liberty": "This text emphasizes freedom, autonomy, rights, and resistance to oppression."
}

prototype_names = list(moral_prototypes.keys())

prototype_embeddings = semantic_model.encode(
    list(moral_prototypes.values()),
    normalize_embeddings=True
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12223.64it/s]


In [58]:
moral_scores = embeddings @ prototype_embeddings.T

moral_df = pd.DataFrame(
    moral_scores,
    columns=[f"moral_{name}" for name in prototype_names]
)

df = pd.concat(
    [df.reset_index(drop=True), moral_df.reset_index(drop=True)],
    axis=1
)

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 768 is different from 384)